# Verification: Real-Time Streaming Architecture

This notebook verifies the new `StreamAnalyzer` class against existing Ground Truth (GT) data.
It simulates a real-time stream by feeding audio chunks (0.5s) into the analyzer.

In [12]:
import numpy as np
import pandas as pd
import librosa
import plotly.graph_objects as go
from pathlib import Path
from tqdm.notebook import tqdm

# Import our new engine
import sys
sys.path.append('../src')
from emotion_stream import StreamAnalyzer, AudioConfig
from mood_mapping import map_mood_tags

In [13]:
# 1. Set up Ground Truth
GT_PRESETS = {
    "china_angry_discuss": [
        {"start": 0, "end": 9, "label": "CALM"},
        {"start": 9, "end": 44, "label": "NOT_CALM"}
    ]
}

FILENAME = "china_angry_discuss.wav"
AUDIO_PATH = Path("../data/audio_samples") / FILENAME

if not AUDIO_PATH.exists():
    # Fallback to current directory if not in subfolder
    AUDIO_PATH = Path(FILENAME)

print(f"Target: {AUDIO_PATH}")

Target: audio_samples/china_angry_discuss.wav


In [14]:
# 2. Run Simulation

def run_simulation(path):
    # Load full audio
    y, sr = librosa.load(path, sr=16000, mono=True)
    
    # Config: 0.5s chunks, 2.0s buffer
    config = AudioConfig(chunk_size=0.5, buffer_size=2.0)
    bot = StreamAnalyzer(config)
    
    chunk_samples = int(config.sr * config.chunk_size)
    results = []
    history = []
    
    print(f"Processing {len(y)/sr:.1f}s audio in {config.chunk_size}s chunks...")
    
    for i in tqdm(range(0, len(y), chunk_samples)):
        chunk = y[i:i+chunk_samples]
        if len(chunk) < chunk_samples:
            chunk = np.pad(chunk, (0, chunk_samples - len(chunk)))
            
        # The Core Step
        analysis = bot.process_chunk(chunk)
        
        # Map to Tags
        tags = map_mood_tags(analysis, history)
        
        # Store
        row = {
            "time_sec": analysis["timestamp"],
            "agitation_score": analysis["agitation_score"],
            "state": analysis["state"],
            **tags # Expand the 9 tags
        }
        results.append(row)
        history.append(analysis)
        
    return pd.DataFrame(results)

if AUDIO_PATH.exists():
    df_res = run_simulation(AUDIO_PATH)
    print(df_res.head())
else:
    print("File not found, skipping simulation.")
    df_res = pd.DataFrame()

Processing 46.0s audio in 0.5s chunks...


  0%|          | 0/93 [00:00<?, ?it/s]

   time_sec  agitation_score     state  sentiment_volatility  \
0       0.0             30.0      CALM                  0.00   
1       0.0             51.0      CALM                  0.00   
2       0.0             65.7  NOT_CALM                 10.50   
3       0.0             76.0  NOT_CALM                 14.65   
4       0.0             83.2  NOT_CALM                 17.28   

  emotional_displays emotional_state_alignment frustration_indicators  \
0               None                   Aligned               Detected   
1               None                Misaligned               Detected   
2               None                Misaligned               Detected   
3               None                Misaligned               Detected   
4         Aggression                Misaligned               Detected   

  frustration_control tone_description sentiment_trend  \
0                High  Loud, Trembling          Stable   
1                High  Loud, Trembling          Rising   
2 

In [15]:
# 3. Plotting Logic (Replicating the Original Viz)

def plot_results(df, filename_stem):
    if df.empty:
        return
        
    gt = GT_PRESETS.get(filename_stem, [])
    
    fig = go.Figure()

    # GT Background
    shapes = []
    annotations = []
    for item in gt:
        color = "rgba(0, 170, 90, 0.5)" if item["label"] == "CALM" else "rgba(220, 40, 30, 0.5)"
        shapes.append({
            "type": "rect",
            "x0": item["start"], "x1": item["end"],
            "y0": 0, "y1": 1, "yref": "paper",
            "fillcolor": color, "line": {"width": 0}, "layer": "below"
        })
        annotations.append({
            "x": (item["start"] + item["end"]) / 2,
            "y": 1.02, "yref": "paper",
            "text": item["label"], "showarrow": False
        })

    # Main Line
    fig.add_trace(go.Scatter(
        x=df["time_sec"], y=df["agitation_score"],
        mode="lines", line=dict(color="#E67E22", width=3),
        name="Agitation (New Stream)"
    ))
    
    # Mood Tags (Hover)
    hover_text = []
    for _, row in df.iterrows():
        t = f"State: {row['state']}<br>Tone: {row['tone_description']}<br>Frust.Control: {row['frustration_control']}"
        hover_text.append(t)
        
    fig.update_traces(text=hover_text, hoverinfo="text+y+x")

    fig.update_layout(
        title=f"Streaming Verification: {filename_stem}",
        shapes=shapes,
        annotations=annotations,
        yaxis=dict(range=[0, 100], title="Agitation Score"),
        xaxis=dict(title="Time (s)"),
        template="plotly_white",
        height=600
    )
    fig.show()

stem = AUDIO_PATH.stem
plot_results(df_res, "china_angry_discuss")

In [16]:
# 4. Check Tags in 'NOT_CALM' zone
if not df_res.empty:
    bad_zone = df_res[df_res["time_sec"] > 15]
    print("Sample Mood Tags in Angry Zone:")
    print(bad_zone[["time_sec", "agitation_score", "tone_description", "frustration_control"]].head(10))

Sample Mood Tags in Angry Zone:
Empty DataFrame
Columns: [time_sec, agitation_score, tone_description, frustration_control]
Index: []
